# ตอนที่ 10: Deployment — คุณไม่ได้รอการคำนวณ คุณรอน้ำหนักเดินทาง

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kobkrit/thai-llm-tutorials/blob/main/notebooks/10_deployment.ipynb)

โน้ตบุ๊กประกอบบทความ **[LLM 10/10] Deployment: คุณไม่ได้รอการคำนวณ คุณรอน้ำหนักเดินทาง**

โดย **ดร.กอบกฤตย์ วิริยะยุทธกร** — ผู้สร้าง OpenThaiGPT, CEO บริษัท iApp Technology
บทความฉบับเต็ม: [kobkrit.com](https://kobkrit.com/blog/llm-10-deployment)

---

## โครงของโน้ตบุ๊กนี้ (เรียงตรงกับหัวข้อในบทความ)

1. ปัญหา (Problem statement)
2. เราจะทำอะไร (Solution)
3. สมการ (Equation) — คำนวณเพดานความเร็วจาก datasheet ก่อนเขียนโค้ด
4. เห็นภาพสมการ (Visualize)
5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline ก่อนแตะอะไรทั้งสิ้น
6. เตรียมข้อมูล (Data) — workload ไม่ใช่ชุดเทรน
7. โค้ดหลัก (Main code) — merge → วัด → ปิดช่องว่างทีละชั้น → quantise
8. ผลลัพธ์ (Results) — ทฤษฎีจาก datasheet เทียบของจริงที่วัดได้
9. เปรียบเทียบ (Comparison) — ตารางสรุป: ทุกความเร็วต้องโชว์ราคาที่จ่าย
10. สรุป (Summary) — และปิดซีรีส์

---

## หัวใจของโน้ตบุ๊กนี้

ตอน decode ทีละ token ที่ batch = 1 การสร้าง token หนึ่งตัวต้อง**อ่านน้ำหนัก
ทุกตัวของโมเดลจาก VRAM หนึ่งรอบเต็ม** แต่ใช้การคำนวณต่อน้ำหนักแค่ ~2 FLOP
GPU จึงนั่งว่างรอข้อมูลเดินทาง — เราจะคำนวณเพดานความเร็ว **ก่อนรันโค้ดแม้แต่บรรทัดเดียว**
(320 GB/s ÷ ~1.2 GB ≈ 266 tok/s) แล้ววัดของจริงมาเทียบ และปิดช่องว่างทีละชั้น
พร้อมกติกาเหล็กของตอน: **ทุกความเร็วที่รายงาน ต้องมีคะแนนคุณภาพ (TH-KNOW ของ
KobEval-TH) แนบมาในแถวเดียวกันเสมอ** — ความเร็วที่ไม่มีคุณภาพกำกับคือโฆษณา

## ก่อนเริ่ม

- โน้ตบุ๊กนี้ออกแบบมาให้รันได้บน **Google Colab แบบฟรี (Tesla T4)**
  เลือก `Runtime > Change runtime type > T4 GPU` ก่อนรันเซลล์แรก
- T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง **ไม่รองรับ bfloat16 และไม่รองรับ FlashAttention-2**
  เราจึงกำหนด `torch_dtype=torch.float16` และ `attn_implementation="sdpa"` เองเสมอ
  และล่วงหน้าไว้เลย: ใน stage 6 vLLM ต้อง `dtype="half"` ด้วยเหตุผลเดียวกัน
- ทุกตอนในซีรีส์นี้ใช้ชุดวัดผลกลางตัวเดียวกันคือ `kobeval` และเบนช์มาร์ก **KobEval-TH**
  ตอนนี้ใช้ slice `TH-KNOW` เป็นคอลัมน์คุณภาพของทุก configuration
- **กติกาการจับเวลา:** อุ่นเครื่อง (warm-up) ก่อนเริ่มนาฬิกาเสมอ ทุก configuration
  ไม่มีข้อยกเว้น — โดยเฉพาะ `torch.compile` ที่ครั้งแรกกินเวลาระดับนาที

## เวลาที่ใช้โดยประมาณบน T4 ฟรี (รวมเวลาโหลดโมเดล)

| ขั้นตอน | เวลาโดยประมาณ |
|---|---|
| Cell 0 ติดตั้ง dependencies | ~2 นาที |
| Cell 1 โหลดโมเดล + baseline TH-KNOW (คอลัมน์คุณภาพ fp16) | ~3 นาที |
| ตรวจเลขคณิต KV / Poisson / Little's law (CPU) | ~เสี้ยววินาที |
| Stage 1 merge adapter + ตรวจ + ขนาดไฟล์ | ~1 นาที |
| Stage 2 baseline tok/s เทียบเพดานทฤษฎี | ~1 นาที |
| Stage 3 static cache + compile + continuous batching | ~5 นาที |
| Stage 4 Poisson load test (p50/p90/p99) | ~3 นาที |
| Stage 5 int8 + nf4 (VRAM, tok/s และ TH-KNOW ทั้งคู่) | ~6 นาที |
| **รวม (ไม่รวม stage 6)** | **~21 นาที** |
| Stage 6 vLLM (ทางเลือก — เปราะที่สุดในซีรีส์) | +5-10 นาที |

ถ้าเวลาไม่พอ: ลด `GEN_TOKENS` หรือ `N_REQ` — และ stage 6 ข้ามได้ทั้ง stage
โดยไม่กระทบข้อสรุปของ stage 1-5 (`results.json` ถูกเขียนก่อนเริ่ม stage 6 เสมอ)


In [ ]:
# =============================================================================
# CELL 0 -- SETUP
# ชุดนี้ต้อง "เหมือนกันทุกตัวอักษร" (byte-identical) ในโน้ตบุ๊กทั้ง 10 ตอน
# ตรวจสอบด้วย: python3 scripts/check_cell0.py
# =============================================================================
# ทำไมต้องเหมือนกันทุกตอน: ถ้า setup ต่างกันแม้นิดเดียว ตัวเลขที่วัดได้จาก
# แต่ละตอนจะเปรียบเทียบกันไม่ได้ ซึ่งทำให้ทั้งซีรีส์นี้ไร้ความหมาย
# -----------------------------------------------------------------------------

# 1) ติดตั้ง dependencies (pin เวอร์ชันไว้ทั้งหมด เพื่อให้ผลลัพธ์ทำซ้ำได้)
#    หมายเหตุ: เราจงใจ "ไม่" ติดตั้ง torch ใหม่ เพราะ Colab มี torch ที่ build
#    มาให้ตรงกับ CUDA driver อยู่แล้ว การ pip install torch ทับคือสาเหตุอันดับ
#    หนึ่งที่ทำให้ notebook พังบน Colab
!pip install -q \
    "transformers==4.51.3" \
    "accelerate==1.6.0" \
    "datasets==3.5.0" \
    "peft==0.15.1" \
    "trl==0.16.1" \
    "bitsandbytes==0.45.5" \
    "sentencepiece==0.2.0" \
    "matplotlib==3.10.1"

# 2) ติดตั้งฟอนต์ไทยให้ matplotlib (ไม่งั้นกราฟจะขึ้นเป็นสี่เหลี่ยม tofu)
!apt-get install -y fonts-thai-tlwg > /dev/null 2>&1

# 3) ดึง repo (ได้ทั้งแพ็กเกจ kobeval และชุดข้อมูล KobEval-TH)
import os
REPO_DIR = "/content/thai-llm-tutorials"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/kobkrit/thai-llm-tutorials.git /content/thai-llm-tutorials
!pip install -q -e /content/thai-llm-tutorials

import random
import numpy as np
import torch

# 4) ยืนยันว่ามี GPU จริง -- ถ้าไม่มีให้หยุดตรงนี้เลย ดีกว่าไปพังตอน train
assert torch.cuda.is_available(), (
    "ไม่พบ GPU! ไปที่ Runtime > Change runtime type > Hardware accelerator > GPU "
    "แล้วรันเซลล์นี้ใหม่"
)

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
# bf16 "แบบมีฮาร์ดแวร์รองรับจริง" เริ่มที่ Ampere (sm_80) ขึ้นไปเท่านั้น
#
# หมายเหตุสำคัญ: torch.cuda.is_bf16_supported() ตอบ True บน T4 (sm_75) ด้วย
# ตั้งแต่ torch รุ่นใหม่ ๆ เพราะมันนับ "การจำลอง (emulation)" ว่ารองรับด้วย
# ซึ่งช้ากว่า fp16 มากและไม่ใช่สิ่งที่เราต้องการ เราจึงเช็ค compute capability
# ตรง ๆ แทน -- นี่คือกับดักจริงที่เจอตอนรันบน Colab
NATIVE_BF16 = CAPABILITY[0] >= 8
SUPPORTS_BF16 = NATIVE_BF16          # ใช้ชื่อเดิมต่อได้ แต่ความหมายคือ "native"

print(f"GPU            : {GPU_NAME}")
print(f"Compute cap.   : sm_{CAPABILITY[0]}{CAPABILITY[1]}")
print(f"VRAM           : {VRAM_GB:.1f} GB")
print(f"SUPPORTS_BF16  : {SUPPORTS_BF16}   (native; torch บอกว่า "
      f"{torch.cuda.is_bf16_supported()} เพราะนับ emulation ด้วย)")
print(f"torch          : {torch.__version__}")

# -----------------------------------------------------------------------------
# 5) จุดสำคัญที่สุดของเซลล์นี้ -- อ่านให้ครบ
#
# ถ้าคุณได้ Tesla T4 (ซึ่งเป็นค่าปกติของ Colab ฟรี) คุณจะเห็น
#     SUPPORTS_BF16 : False   (native)
# ทั้งที่ torch.cuda.is_bf16_supported() ตอบ True -- อย่าเชื่อค่านั้น
#
# T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง "ไม่รองรับ" สองอย่างนี้:
#   - bfloat16  -> ต้องใช้ float16 แทน
#   - FlashAttention-2 -> ต้องใช้ sdpa แทน
#
# กับดักที่คนเจอบ่อยที่สุด: config ของ Qwen3-0.6B ระบุ torch_dtype: bfloat16
# ไว้ในไฟล์ ดังนั้นถ้าเขียน torch_dtype="auto" transformers จะเชื่อ config
# แล้วโหลดเป็น bf16 บนการ์ดที่ไม่รองรับ ผลคือได้ NaN, loss ไม่ลด หรือ
# โมเดลพ่นข้อความมั่ว โดยไม่มี error ฟ้องให้เห็นเลย
#
# เราจึงกำหนดค่าพวกนี้เอง "อย่างชัดเจน" ทุกครั้ง ไม่พึ่ง auto
# -----------------------------------------------------------------------------
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
ATTN_IMPL = "sdpa"          # ห้ามใช้ flash_attention_2 บน T4
USE_FP16 = not SUPPORTS_BF16  # ส่งเข้า TrainingArguments: fp16=USE_FP16, bf16=SUPPORTS_BF16

print(f"\n--> DTYPE      : {DTYPE}")
print(f"--> ATTN_IMPL  : {ATTN_IMPL}")
print(f"--> fp16={USE_FP16}, bf16={SUPPORTS_BF16}  (ใช้คู่นี้กับ TrainingArguments)")

# 6) ตั้ง seed ทุกตัวที่เกี่ยวข้อง เพื่อให้ผลลัพธ์ทำซ้ำได้
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# 7) import kobeval -- ชุดวัดผลกลางที่ใช้เหมือนกันทั้ง 10 ตอน
#    pip install -e ลงทะเบียนแพ็กเกจไว้แล้ว แต่ sys.path ของเคอร์เนลที่รันอยู่
#    อาจยังไม่เห็น จึงใส่ path ของ repo เข้าไปตรง ๆ กันพลาด (idempotent)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from kobeval import evaluate, compare, plot_before_after, th_ratio, wilson_ci
from kobeval import EVAL_CONTRACT, verify_checksums

print(f"\nkobeval contract: {EVAL_CONTRACT}")
print(f"KobEval-TH checksum ok: {verify_checksums()['ok']}")


---

## 1. ปัญหา (Problem statement)

โมเดลผ่านการเทรนและวัดผลมาครบ 9 ตอน คำถามที่เหลือไม่ใช่คำถาม machine learning:

| คำถามของคนจ่ายเงิน | ตัวเลขที่ตอบ |
|---|---|
| ผู้ใช้หนึ่งคนต้องรอนานแค่ไหน | p50 / p99 latency |
| รับผู้ใช้พร้อมกันได้กี่คน | concurrency (Little's law) |
| ต้องใช้ GPU กี่ตัว | throughput (tok/s) |
| ให้ context ยาวได้แค่ไหน | งบ KV cache |

และตัวเลข benchmark ที่เจอตามบทความมักไร้ความหมายเพราะ: รายงาน tok/s
โดยไม่บอก batch size, เอา prefill มาเฉลี่ยรวมกับ decode, และรายงานความเร็ว
หลัง quantise **โดยไม่รายงานคุณภาพ** — บาปต้นของ genre นี้

---

## 2. เราจะทำอะไร (Solution)

1. **คำนวณเพดาน** ความเร็ว decode จาก datasheet ของ T4 — ยังไม่ต้องรันอะไรเลย
2. **วัดของจริง** ด้วย `generate` ตรง ๆ แล้วดูว่าห่างเพดานกี่เท่า
3. **ปิดช่องว่าง** ทีละขั้น — static KV cache + `torch.compile`,
   แล้ว continuous batching ที่เขียนเองราว 60 บรรทัด — วัดใหม่ทุกขั้น
4. **จ่ายด้วยอะไร** — quantise เป็น int8 และ nf4 แล้ววัดทั้ง VRAM, ความเร็ว
   *และ* คุณภาพบน TH-KNOW คู่กันเสมอ
5. **โหลดจริง** — ยิง request แบบ Poisson (open-loop) แล้วดู p99 ระเบิดก่อนเครื่องเต็ม

---

## 3. สมการ (Equation)

### 3.1 งบหน่วยความจำตอนเสิร์ฟ

$$M = \underbrace{P\,b_w}_{\text{weights}} + \underbrace{2\,L\,n_{kv}\,d_h\,s\,B\,b_{kv}}_{\text{KV cache}} + M_{\text{act}}$$

แทนค่าจริงจาก `config.json` ของ Qwen3-0.6B ($L=28$, $n_{kv}=8$, $d_h=128$, fp16 = 2 ไบต์):

$$\text{KV/token} = 2 \times 28 \times 8 \times 128 \times 2 = 114{,}688 \text{ ไบต์} = 112\ \text{KiB พอดีเป๊ะ}$$

กับดักที่พลาดบ่อยที่สุด: Qwen3 ใช้ grouped-query attention ต้องใช้ $n_{kv}=8$
ไม่ใช่จำนวน attention heads (16) — ใช้ผิดตัวเดียว คำตอบคูณสองทันที
เลขนี้จะถูก `assert` จากค่า config จริงในเซลล์ถัดไป ไม่ใช่เลขที่จำมา
ที่ context เต็ม 40,960 token มันคือ ~4.7 GB **ต่อ sequence เดียว** —
เกือบ 4 เท่าของน้ำหนักโมเดล: **context คือตัวกินงบ ไม่ใช่โมเดล**

### 3.2 สมการที่สำคัญที่สุดของตอน — เพดานของ decode

$$\text{tok/s} \lesssim \frac{\text{BW}}{M_{\text{weights}}} = \frac{320\ \text{GB/s}}{\approx 1.2\ \text{GB}} \approx 266$$

320 GB/s คือแบนด์วิดท์ GDDR6 จาก datasheet ของ T4 ตรง ๆ — ไม่มีโค้ดใดทำให้ T4
decode โมเดลนี้แบบ single-stream เร็วกว่านี้ได้ เพราะมันคือขีดจำกัดของสายไฟ
เช็คว่าคอขวดคือแบนด์วิดท์จริง: ที่ 266 tok/s งานคำนวณ ~0.32 TFLOPS
คือ **ราว 0.5%** ของ 65 TFLOPS (fp16) — ชิปว่างงาน 99.5%

### 3.3 Little's Law

$$L = \lambda W$$

จำนวนงานค้างในระบบ = อัตรางานเข้า × เวลาเฉลี่ยต่องาน — จริงเสมอ ไม่มีสมมติฐาน
เรื่องการแจกแจง และมันผูกกลับไปที่สมการ 3.1: ถือ $L$ sequence = จองงบ KV ตาม $L$

### 3.4 INT8 symmetric quantisation

$$s = \frac{\max|x|}{127},\qquad x_q = \mathrm{round}(x/s),\qquad \hat x = s\,x_q$$

ไบต์ต่อน้ำหนักลดครึ่ง — สมการ 3.2 บอกว่า**ในทางทฤษฎี** decode เร็วขึ้น 2 เท่า
ในทางปฏิบัติ kernel ที่ต้อง dequantise อาจกินกำไรหมดหรือเกิน (โดยเฉพาะ
LLM.int8() บน T4) — ต้องวัด ห้ามเดา และ KV cache ยังเป็น fp16 ที่ 112 KiB/token เท่าเดิม

### 3.5 Prefill กับ Decode — ห้ามเอามาปนกัน

จุดสัน (ridge point) ของ T4: $65\ \text{TFLOPS} / 320\ \text{GB/s} \approx 203$ FLOP/byte
— decode ที่ batch 1 มี intensity ~1 (bandwidth-bound ต่ำกว่าจุดสัน 200 เท่า)
ส่วน prefill ของ prompt ยาว $s$ มี intensity ~$s$ (compute-bound ตั้งแต่ ~200 token)
โน้ตบุ๊กนี้จึงรายงาน **TTFT** (วัด prefill) กับ **ITL** (วัด decode) แยกกันเสมอ

---

## 4. เห็นภาพสมการ (Visualize)

กราฟของหัวข้อนี้ (roofline และ latency knee) ถูกวาด**จากตัวเลขที่วัดจริง**
ในหัวข้อ 8-9 แทนที่จะวาดจากตัวเลขสมมติล่วงหน้า — กติกาของซีรีส์คือ
เซลล์โค้ดที่สองต้องเป็น baseline เสมอ ห้ามมีอะไรแทรกก่อน


---

## 5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline

หลักการของทั้งซีรีส์: **วัดก่อนเสมอ**

เซลล์ถัดไปคือ Cell 1 โหลด `Qwen/Qwen3-0.6B` แล้ววัด KobEval-TH slice `TH-KNOW`
ตัวเลขนี้คือ **คอลัมน์คุณภาพของแถว fp16** ในตารางสรุปหัวข้อ 9 — ทุก configuration
ที่เราจะวัดความเร็ว (compile, int8, nf4, vLLM) ต้องถูกเทียบคุณภาพกลับมาที่เลขนี้

สังเกตค่า `th_ratio` ตามเคย — โมเดลที่ quantise แล้วเริ่มตอบภาษาอังกฤษปน
คือความเสียหายที่ accuracy อย่างเดียวมองไม่เห็น

> **หมายเหตุเรื่องเวลา:** ใช้ slice เดียว (TH-KNOW, 30 ข้อ) เพราะตอนนี้ต้องวัดซ้ำ
> ถึง 3 รอบ (fp16 / int8 / nf4) รอบละ ~3 นาที การใช้ครบ 4 slice จะเกินงบ T4 ฟรี


In [ ]:
# =============================================================================
# CELL 1 -- BASELINE (วัดผลก่อนแตะอะไรทั้งสิ้น)
# เซลล์นี้เป็นเซลล์โค้ดที่สองของทุกตอนในซีรีส์
# =============================================================================
import gc
import json
import math
import os
import time

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
KOBEVAL_SLICES = ["TH-KNOW"]     # คอลัมน์คุณภาพของทุกแถวในตารางสรุป
DEV = "cuda:0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,             # <-- ห้ามใช้ค่า auto เด็ดขาดบน T4 (ดูคอมเมนต์ Cell 0)
    attn_implementation=ATTN_IMPL, # <-- sdpa เท่านั้น
    device_map=DEV,
)
model.eval()

cfg = model.config
N_PARAMS = sum(p.numel() for p in model.parameters())
WEIGHT_BYTES = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"โหลดโมเดลแล้ว: {MODEL_ID}")
print(f"dtype จริงของ parameter: {next(model.parameters()).dtype}")
print(f"จำนวนพารามิเตอร์: {N_PARAMS / 1e6:.1f}M  -> น้ำหนัก {WEIGHT_BYTES / 1e9:.2f} GB (fp16)")
print("--- ค่าจริงจาก config ของ Qwen3-0.6B (สมการ 3.1 จะใช้ตัวเลขพวกนี้) ---")
print(f"  num_hidden_layers       : {cfg.num_hidden_layers}")
print(f"  num_attention_heads     : {cfg.num_attention_heads}")
print(f"  num_key_value_heads     : {cfg.num_key_value_heads}  <- GQA: ใช้ตัวนี้ ไม่ใช่ 16")
print(f"  head_dim                : {cfg.head_dim}")
print(f"  max_position_embeddings : {cfg.max_position_embeddings}")

# รันเบนช์มาร์กกลาง -- สัญญาการวัดผลถูกตรึงไว้ใน kobeval แล้ว
# (greedy, max_new_tokens=256, enable_thinking=False, seed=42)
t0 = time.time()
baseline = evaluate(
    model,
    tokenizer,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name="Qwen3-0.6B fp16 (baseline)",
    out_path="results_baseline.json",
)
print(f"\nใช้เวลาวัด baseline: {time.time() - t0:.0f} วินาที")

for name, s in baseline["slices"].items():
    print(
        f"{name:9s} acc={s['accuracy']:.3f} "
        f"[95% CI {s['ci_low']:.3f}-{s['ci_high']:.3f}]  "
        f"th_ratio={s['th_ratio']:.2f}  len={s['mean_output_len']:.0f}"
    )

FP16_QUALITY = baseline["slices"]["TH-KNOW"]
print("\nจดตัวเลขนี้ไว้: มันคือคอลัมน์ TH-KNOW ของแถว fp16 ในตารางสรุปหัวข้อ 9")
print("ทุกความเร็วที่ได้มาหลังจากนี้ ต้องเอาคุณภาพมาวางเทียบกับมันเสมอ")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 3 (ตรวจด้วยโค้ด) -- เลขคณิต KV, เพดาน tok/s, Poisson, Little's law
# สามอย่างแรกเป็นคณิตศาสตร์ล้วน: รันได้แม้ไม่มี GPU
# -----------------------------------------------------------------------------
import numpy as np

# --- (1) KV cache ต่อ token จากค่า config จริง (สมการ 3.1) ---------------------
KV_PER_TOKEN = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * cfg.head_dim * 2
print(f"KV/token = 2 x {cfg.num_hidden_layers} x {cfg.num_key_value_heads} "
      f"x {cfg.head_dim} x 2 ไบต์ = {KV_PER_TOKEN:,} ไบต์ = {KV_PER_TOKEN / 1024:.0f} KiB")
assert KV_PER_TOKEN == 114_688, "ต้องได้ 114,688 ไบต์ (112 KiB) พอดีจาก config จริง"
assert KV_PER_TOKEN == 112 * 1024

kv_full_ctx = KV_PER_TOKEN * cfg.max_position_embeddings
print(f"ที่ context เต็ม {cfg.max_position_embeddings:,} token: "
      f"KV = {kv_full_ctx / 1e9:.2f} GB ต่อ 1 sequence "
      f"(~{kv_full_ctx / WEIGHT_BYTES:.1f} เท่าของน้ำหนักโมเดล)")

# กับดัก GQA: ถ้าเผลอใช้ num_attention_heads (16) แทน num_key_value_heads (8)
kv_wrong = 2 * cfg.num_hidden_layers * cfg.num_attention_heads * cfg.head_dim * 2
print(f"ถ้าใช้ n_heads ผิดตัว จะได้ {kv_wrong:,} ไบต์ -- คูณสองพอดี ({kv_wrong // KV_PER_TOKEN}x)")

# --- (2) เพดาน decode จาก datasheet (สมการ 3.2) -------------------------------
T4_BW = 320e9                    # GDDR6 bandwidth ของ T4 จาก datasheet (ไบต์/วินาที)
CEILING = T4_BW / WEIGHT_BYTES   # อ่านน้ำหนักทุกตัวหนึ่งรอบต่อ 1 token
print(f"\nเพดานทฤษฎี batch=1: {T4_BW / 1e9:.0f} GB/s / {WEIGHT_BYTES / 1e9:.2f} GB "
      f"= {CEILING:.0f} tok/s")
assert 240 < CEILING < 300, "Qwen3-0.6B fp16 บน T4 ต้องได้เพดานราว 266 tok/s"
flops_at_ceiling = 2 * N_PARAMS * CEILING
print(f"งานคำนวณที่เพดาน: {flops_at_ceiling / 1e12:.2f} TFLOPS "
      f"จาก 65 TFLOPS ของ T4 = ใช้ชิปแค่ {100 * flops_at_ceiling / 65e12:.1f}%")

# --- (3) Poisson generator: อัตราเฉลี่ยของ 10,000 ตัวอย่างต้องตรง lambda ----------
rng = np.random.default_rng(SEED)
lam_test = 4.0
gaps = rng.exponential(1.0 / lam_test, size=10_000)
measured_rate = 1.0 / gaps.mean()
print(f"\nPoisson check: lambda ที่ตั้ง = {lam_test:.2f} req/s, "
      f"วัดจาก 10,000 ตัวอย่าง = {measured_rate:.3f} req/s "
      f"(คลาด {100 * abs(measured_rate - lam_test) / lam_test:.2f}%)")
assert abs(measured_rate - lam_test) / lam_test < 0.03, "อัตราเฉลี่ยต้องคลาดไม่เกิน 3%"

# --- (4) Little's law บนตัวเลขสังเคราะห์: L = lambda x W --------------------------
lam_ll, W_ll = 5.0, 2.0                         # 5 req/s, งานละ 2 วินาที
arrivals = np.cumsum(rng.exponential(1.0 / lam_ll, size=10_000))
horizon = arrivals[-1] + W_ll
L_measured = (len(arrivals) * W_ll) / horizon    # เวลารวมที่งานอยู่ในระบบ / เวลาทั้งหมด
print(f"Little's law: lambda={lam_ll} req/s, W={W_ll}s -> ทฤษฎี L={lam_ll * W_ll:.0f} "
      f"| จากการจำลอง L={L_measured:.2f}")
assert abs(L_measured - lam_ll * W_ll) < 0.3, "L ต้องลู่เข้า lambda*W"

print("\nผ่านทุกข้อ: สมการ 3.1-3.3 ไม่ใช่เลขในบทความ มันคือ assert ที่เพิ่งรันจริง")
print(f"ที่ lambda=5, W=2: ต้องถือ {lam_ll * W_ll:.0f} sequence พร้อมกัน = จอง KV "
      f"{lam_ll * W_ll * 1024 * KV_PER_TOKEN / 1e9:.2f} GB ที่ context เฉลี่ย 1,024 token")


---

## 7. โค้ดหลัก (Main code)

### Stage 1 — merge LoRA adapter กลับเข้าน้ำหนักฐาน

ตลอดซีรีส์เราเทรนด้วย LoRA ซึ่งตอนเสิร์ฟกลายเป็นภาระ: ทุก forward ต้องคูณ $BAx$
เพิ่มอีกทอด ข่าวดีคือ LoRA merge ได้แบบปิดรูป: $W' = W + \tfrac{\alpha}{r}BA$
— เสิร์ฟแล้วต้นทุนเท่าโมเดลฐานเป๊ะ และทุกเครื่องมือเสิร์ฟ (รวม vLLM)
มองเห็นมันเป็นโมเดลธรรมดาหนึ่งก้อน

เซลล์ถัดไปมองหา adapter ที่ดีที่สุดจากตอนก่อน ๆ ตามธรรมเนียม `checkpoints/`
(ลำดับ: DPO จากตอนที่ 4 → SFT จากตอนที่ 2) ถ้าไม่เจอ ใช้โมเดลฐานเปล่า ๆ
เป็นโมเดลเสิร์ฟแทน — ทุกการวัดที่เหลือทำงานเหมือนกันทุกประการ

**อย่าเชื่อว่า merge แล้วได้โมเดลเดิม — พิสูจน์** ด้วยการเทียบ logits ก่อน/หลัง merge:
ค่าต่างระดับ ~1e-3 คือการปัดเศษ fp16 ปกติ ถ้าเจอระดับ 1.0 แปลว่าโหลด adapter
ผิดตัวหรือ dtype ไม่ตรง


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 1 -- merge_and_unload adapter ที่ดีที่สุด (หรือใช้ base ถ้าไม่มี)
# -----------------------------------------------------------------------------
from peft import PeftModel

SERVE_DIR = "qwen3-th-serve"
ADAPTER_CANDIDATES = [                    # เรียงตามลำดับความน่าใช้เสิร์ฟ
    "checkpoints/04_dpo_lora", "qwen3-0.6b-th-dpo-lora",
    "checkpoints/02_sft_lora", "qwen3-0.6b-th-sft-lora",
]

t_stage1 = time.time()
adapter_path = next((p for p in ADAPTER_CANDIDATES if os.path.isdir(p)), None)
probe = tokenizer("การไฟฟ้าส่วนภูมิภาคมีหน้าที่อะไร", return_tensors="pt").to(DEV)

if adapter_path is not None:
    print(f"พบ adapter: {adapter_path} -> โหลดฐานสำเนาใหม่แล้ว merge")
    base2 = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
        attn_implementation=ATTN_IMPL,
        device_map=DEV,
    )
    policy = PeftModel.from_pretrained(base2, adapter_path)
    policy.eval()

    adapter_bytes = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(adapter_path) for f in files)

    with torch.no_grad():
        logits_pre = policy(**probe).logits.float()
    merged = policy.merge_and_unload()               # W' = W + (alpha/r) B A
    merged.eval()
    with torch.no_grad():
        logits_post = merged(**probe).logits.float()

    dmax = (logits_pre - logits_post).abs().max().item()
    print(f"max |delta logit| ก่อน/หลัง merge = {dmax:.5f}")
    print("  ~1e-3 = การปัดเศษ fp16 ปกติ | ~1.0 = โหลด adapter ผิดตัวหรือ dtype ไม่ตรง")
    assert dmax < 0.05, "merge เปลี่ยนพฤติกรรมโมเดลเกินระดับปัดเศษ -- หยุดตรวจก่อน"
    print(f"ขนาด adapter บนดิสก์: {adapter_bytes / 1e6:.1f} MB")
else:
    print(f"ไม่พบ adapter ใน {ADAPTER_CANDIDATES}")
    print("-> ใช้โมเดลฐานเป็นโมเดลเสิร์ฟ (ทุกการวัดที่เหลือทำงานเหมือนกันทุกประการ)")
    merged = model            # โมเดลจาก Cell 1
    adapter_bytes = None

merged.save_pretrained(SERVE_DIR)
tokenizer.save_pretrained(SERVE_DIR)
serve_bytes = sum(os.path.getsize(os.path.join(dp, f))
                  for dp, _, files in os.walk(SERVE_DIR) for f in files)
print(f"เซฟโมเดลเสิร์ฟแล้วที่ {SERVE_DIR}/ ({serve_bytes / 1e9:.2f} GB)")
if adapter_bytes:
    print(f"ราคาของการ merge: ไฟล์ {adapter_bytes / 1e6:.0f} MB กลายเป็น "
          f"{serve_bytes / 1e9:.2f} GB (~{serve_bytes / adapter_bytes:.0f} เท่า) "
          f"แลกกับ inference ที่ไม่มีภาระ BAx อีกต่อไป")

# คืน VRAM ของสำเนาที่ไม่ใช้แล้ว (ถ้า merge สำเร็จ โมเดลของ Cell 1 ปลดได้)
if adapter_path is not None:
    del model, policy, base2
    gc.collect()
    torch.cuda.empty_cache()
    model = merged            # ให้ตัวแปรเดียวชี้โมเดลเสิร์ฟเสมอ
print(f"Stage 1 ใช้เวลา {time.time() - t_stage1:.0f} วินาที")


---

## 6. เตรียมข้อมูล (Data) — workload ไม่ใช่ชุดเทรน

"ข้อมูล" ของตอนนี้คือ **ภาระงาน (workload)** สามส่วน:

1. **พรอมป์ตภาษาไทย** จากคำถาม TH-KNOW ของ KobEval-TH — สั้น/ยาวคละกัน
   ให้ prefill มีความหลากหลายแบบงานจริง (ใช้เป็นพรอมป์ตวัดความเร็วเท่านั้น
   ไม่ใช่การวัดคุณภาพ — คุณภาพวัดผ่าน `evaluate()` ตามสัญญาเสมอ)
2. **ชุดวัดคุณภาพ TH-KNOW** — วัดซ้ำกับทุก configuration ที่แตะน้ำหนัก (int8, nf4)
3. **ตัวยิงโหลดแบบ open-loop** — ตารางเวลามาถึงสุ่มแบบ Poisson แล้วยิง**ตามนัด**
   ไม่ว่าเครื่องจะพร้อมหรือไม่

> **ทำไมต้อง open-loop — กับดักชื่อ coordinated omission:**
> ถ้ายิงทีละคำขอแล้ว*รอจนได้คำตอบ*ก่อนยิงต่อ (closed-loop) เซิร์ฟเวอร์ที่ช้า
> จะทำให้เครื่องมือวัดยิงช้าลงเอง คิวไม่มีวันสะสม และ p99 จะสวยหลอก ๆ
> การยิงตามตารางเวลาที่สุ่มไว้ล่วงหน้าคือวิธีเดียวที่ "หัวเข่า" ของ latency
> จะโผล่ให้เห็นจริงใน stage 4

### Stage 2 — baseline `generate` เทียบเพดานจาก datasheet

วัด decode แบบตรงไปตรงมาที่สุด (batch 1, `generate` ธรรมดา) พร้อมกติกาเหล็ก:
**อุ่นเครื่องก่อนจับเวลา** และแยก TTFT (prefill) ออกจาก ITL (decode) เสมอ
ตัวเลขที่ได้จะต่ำกว่าเพดาน ~266 tok/s หลายเท่า — อย่าตกใจ และอย่าโทษ T4
ช่องว่างนั้นคือ overhead ของ Python ต่อ step ซึ่ง stage 3 จะกำจัดทีละชั้น


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 2 -- baseline tok/s (batch 1) เทียบเพดานทฤษฎี + แยก TTFT / ITL
# -----------------------------------------------------------------------------
from kobeval import load_slice

GEN_TOKENS = 128          # จำนวน token ที่ decode ต่อการวัดหนึ่งครั้ง
WARMUP_RUNS = 2           # กติกาเหล็ก: อุ่นเครื่องก่อนจับเวลาเสมอ

know_items = load_slice("TH-KNOW")
PROMPTS = [it["prompt"] for it in know_items]      # พรอมป์ตไทยคละความยาว
RESULTS_SPEED = {}                                  # ชื่อ config -> ตัวเลขที่วัดได้


def timed_decode(m, prompt, n_new=GEN_TOKENS, warmup=WARMUP_RUNS):
    """คืน (tok/s ของช่วง decode, TTFT วินาที) -- แยก prefill ออกจาก decode"""
    ids = tokenizer(prompt, return_tensors="pt").to(m.device)
    for _ in range(warmup):                        # อุ่นเครื่อง: ไม่จับเวลา
        with torch.no_grad():
            m.generate(**ids, max_new_tokens=8, do_sample=False,
                       pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize()

    t0 = time.time()                               # TTFT: prefill + token แรก
    with torch.no_grad():
        m.generate(**ids, max_new_tokens=1, do_sample=False,
                   pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize()
    ttft = time.time() - t0

    t0 = time.time()
    with torch.no_grad():
        out = m.generate(**ids, max_new_tokens=n_new, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize()
    total = time.time() - t0
    n_generated = out.shape[1] - ids["input_ids"].shape[1]
    itl = (total - ttft) / max(1, n_generated - 1)  # inter-token latency
    return n_generated / total, ttft, itl


tok_s, ttft, itl = timed_decode(model, PROMPTS[0])
RESULTS_SPEED["fp16 naive generate"] = {"tok_s_b1": tok_s, "ttft_s": ttft, "itl_ms": 1000 * itl}

print(f"เพดานทฤษฎีจาก datasheet : {CEILING:6.0f} tok/s   (สมการ 3.2)")
print(f"วัดจริง naive generate  : {tok_s:6.1f} tok/s   (batch 1, {GEN_TOKENS} token)")
print(f"อัตราส่วน วัดจริง/ทฤษฎี  : {tok_s / CEILING:.2f}  "
      f"(ห่างเพดาน {CEILING / tok_s:.1f} เท่า)")
print(f"TTFT = {1000 * ttft:.0f} ms (prefill, compute-bound) | "
      f"ITL = {1000 * itl:.1f} ms/token (decode, bandwidth-bound)")
print()
print("ช่องว่างจากเพดานไม่ใช่ความผิดของ T4 -- มันคือ Python วนลูปต่อ token,")
print("การปล่อย kernel นับสิบครั้งต่อ step และการจอง KV แบบ dynamic")
print("stage 3 จะกำจัดทีละชั้นแล้ววัดใหม่ทุกครั้ง")


### Stage 3a — static KV cache + `torch.compile`

`generate` ปกติขยาย KV cache ทีละ token ทำให้ shape เปลี่ยนตลอดและ compile ไม่ได้
จอง cache เต็มก้อนล่วงหน้า (static) แล้ว shape จะนิ่งพอให้ `torch.compile`
จับกราฟลง CUDA graph — กำจัด overhead ตัวใหญ่ที่สุดของ baseline
คือการปล่อย kernel ทีละตัวจาก Python

> **ไม่อุ่นเครื่องก่อนวัด = ตัวเลขทั้งชุดเป็นขยะ:** การเรียกครั้งแรกของโมเดล
> ที่ compile แล้วใช้เวลา**หลายสิบวินาทีถึงระดับนาที**ในการ trace + compile
> ถ้าเวลานั้นปนเข้าการจับเวลา คุณจะสรุปว่า compile "ทำให้ช้าลง" ซึ่งกลับด้าน
> กับความจริง — เซลล์นี้รันทิ้ง 3 รอบก่อนเริ่มนาฬิกา

บน Colab บางรุ่น `torch.compile` อาจล้มเหลว (เวอร์ชัน torch/triton ไม่ลงตัว)
เซลล์นี้จึงกันไว้ด้วย try/except: ถ้า compile ไม่ได้ จะวัดแบบ static cache
อย่างเดียวแล้วบอกตรง ๆ


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 3a -- static KV cache + torch.compile (อุ่นเครื่อง 3 รอบก่อนจับเวลา)
# -----------------------------------------------------------------------------
t_stage3 = time.time()
model.generation_config.cache_implementation = "static"

COMPILE_OK = False
try:
    fast = torch.compile(model, mode="reduce-overhead")
    warm = tokenizer("อุ่นเครื่องก่อนจับเวลาเสมอ", return_tensors="pt").to(DEV)
    for i in range(3):                     # compile จริงเกิดตรงนี้ -- ช้าและตั้งใจให้ช้า
        t0 = time.time()
        with torch.no_grad():
            fast.generate(**warm, max_new_tokens=8, do_sample=False,
                          pad_token_id=tokenizer.pad_token_id)
        print(f"  warm-up รอบ {i + 1}: {time.time() - t0:.1f}s "
              + ("(รอบแรกช้าที่สุด = ค่า compile)" if i == 0 else ""))
    COMPILE_OK = True
except Exception as exc:
    print(f"torch.compile ใช้ไม่ได้บนรันไทม์นี้ ({exc}) -> วัดแบบ static cache อย่างเดียว")
    fast = model

tok_s_c, ttft_c, itl_c = timed_decode(fast, PROMPTS[0])
tag = "fp16 + static cache + compile" if COMPILE_OK else "fp16 + static cache"
RESULTS_SPEED[tag] = {"tok_s_b1": tok_s_c, "ttft_s": ttft_c, "itl_ms": 1000 * itl_c}

base_speed = RESULTS_SPEED["fp16 naive generate"]["tok_s_b1"]
print(f"\n{tag}: {tok_s_c:.1f} tok/s (batch 1)")
print(f"เทียบ naive {base_speed:.1f} tok/s -> เร็วขึ้น {tok_s_c / base_speed:.2f} เท่า")
print(f"เทียบเพดานทฤษฎี {CEILING:.0f} tok/s -> ตอนนี้ได้ {100 * tok_s_c / CEILING:.0f}%")
print(f"Stage 3a ใช้เวลา {time.time() - t_stage3:.0f} วินาที (ส่วนใหญ่คือค่า compile)")

# ปิด static cache สำหรับ stage ถัดไป -- continuous batching จัดการ cache เอง
model.generation_config.cache_implementation = None


### Stage 3b — continuous batching เขียนเองราว 60 บรรทัด

สมการ 3.2 บอกว่าการอ่านน้ำหนักหนึ่งรอบคือต้นทุนก้อนใหญ่ — batching คือการหาร
token หลายตัวด้วยต้นทุนก้อนเดียวกัน แต่ static batching (รอให้ครบ $B$ ค่อยเริ่ม
แล้วรอตัวที่ยาวที่สุดจบ) ทิ้งที่ว่างมหาศาล **continuous batching** จึงรับ request
ใหม่เข้า batch ทันทีที่มีที่ว่าง และปล่อยตัวที่จบออกทันที

โครงของเรา (ตัวจริงอยู่ในเซลล์ถัดไป ราว 60 บรรทัดรวมคอมเมนต์):

- รับเข้า: เมื่อสมาชิกของ batch เปลี่ยน ให้ **prefill ใหม่ทั้ง batch** ด้วย
  left-padding (จ่าย prefill ซ้ำ — นี่คือราคาที่ vLLM แก้ด้วย paged KV)
- ระหว่างสมาชิกคงที่: decode ก้าวละ 1 token ให้ทุก sequence พร้อมกัน
  ด้วย KV cache ก้อนเดียว — อ่านน้ำหนัก 1 รอบ ได้ $B$ token
- ตัวที่เจอ `eos` หรือครบโควตา ออกจาก batch ทันที คืนที่ให้คิว

มันไม่ใช่ vLLM — ไม่มี paged memory, ไม่มี prefix cache — แต่มันพิสูจน์กลไก
ด้วยโค้ดที่อ่านจบในหนึ่งหน้าจอ และการปรับปรุงที่วัดได้ของมันอธิบายที่มาได้ทุกเปอร์เซ็นต์


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 3b -- continuous batching ~60 บรรทัด แล้ววัดที่ batch 1/4/8/16
# -----------------------------------------------------------------------------
from collections import deque


class Seq:
    """สถานะของหนึ่ง request ในระบบ"""

    def __init__(self, prompt, max_new, arrival=0.0):
        self.ids = tokenizer(prompt, add_special_tokens=True).input_ids
        self.out = []
        self.max_new = max_new
        self.arrival = arrival          # เวลาที่ request มาถึง (ใช้ใน stage 4)
        self.t_done = None

    @property
    def done(self):
        return (len(self.out) >= self.max_new
                or (self.out and self.out[-1] == tokenizer.eos_token_id))


@torch.no_grad()
def continuous_batching(queue, max_batch, clock=None):
    """สเกดูลเลอร์หลัก: prefill ใหม่เมื่อสมาชิกเปลี่ยน / decode ร่วมกันเมื่อคงที่

    queue: deque ของ Seq (เรียงตามเวลามาถึง) | clock: ฟังก์ชันเวลา (stage 4)
    คืน (จำนวน token ทั้งหมดที่สร้าง, เวลาที่ใช้)
    """
    running, cache, total_new = [], None, 0
    t_start = time.time()
    now = clock or (lambda: time.time() - t_start)

    while queue or running:
        # --- รับเข้า: หยิบจากคิวทุกตัวที่ "มาถึงแล้ว" และยังมีที่ว่าง -----------------
        changed = False
        while queue and len(running) < max_batch and queue[0].arrival <= now():
            running.append(queue.popleft())
            changed = True
        if not running:                       # คิวยังไม่ถึงเวลา -- รอสั้น ๆ แล้วเช็คใหม่
            time.sleep(0.002)
            continue

        if changed or cache is None:          # --- prefill ใหม่ทั้ง batch (left-pad) --
            seqs = [s.ids + s.out for s in running]
            width = max(len(s) for s in seqs)
            pad = tokenizer.pad_token_id
            ids = torch.tensor([[pad] * (width - len(s)) + s for s in seqs], device=DEV)
            mask = torch.tensor([[0] * (width - len(s)) + [1] * len(s) for s in seqs],
                                device=DEV)
            # left-pad ต้องส่ง position_ids เอง ไม่งั้น RoPE นับตำแหน่ง pad รวมไปด้วย
            pos = (mask.cumsum(-1) - 1).clamp(min=0)
            out = model(input_ids=ids, attention_mask=mask, position_ids=pos,
                        use_cache=True)
            cache, att_mask, last_pos = out.past_key_values, mask, pos[:, -1]
        else:                                 # --- decode ก้าวเดียวให้ทั้ง batch -------
            step_ids = torch.tensor([[s.out[-1]] for s in running], device=DEV)
            att_mask = torch.cat([att_mask, torch.ones_like(step_ids)], dim=1)
            last_pos = last_pos + 1
            out = model(input_ids=step_ids, attention_mask=att_mask,
                        position_ids=last_pos.unsqueeze(-1),
                        past_key_values=cache, use_cache=True)
            cache = out.past_key_values

        next_tok = out.logits[:, -1].argmax(-1)          # greedy ต่อ sequence
        for s, t in zip(running, next_tok.tolist()):
            s.out.append(t)
            total_new += 1

        finished = [s for s in running if s.done]
        for s in finished:
            s.t_done = now()
        if finished:                          # สมาชิกเปลี่ยน -> รอบหน้า prefill ใหม่
            running = [s for s in running if not s.done]
            cache = None

    return total_new, time.time() - t_start


# --- วัดที่ batch 1/4/8/16: ส่ง 16 request พร้อมกัน (arrival=0) --------------------
N_JOBS, JOB_TOKENS = 16, 64
print(f"งาน {N_JOBS} request x {JOB_TOKENS} token | อ่านน้ำหนัก 1 รอบได้ B token\n")
print(f"{'max_batch':>10s} {'aggregate tok/s':>16s} {'ต่อ request (วินาที)':>22s}")

BATCH_SWEEP = {}
for B in (1, 4, 8, 16):
    q = deque(Seq(PROMPTS[i % len(PROMPTS)], JOB_TOKENS) for i in range(N_JOBS))
    jobs = list(q)
    if B == 1:                                # อุ่นเครื่องรอบแรกด้วยงานเล็ก
        wq = deque([Seq(PROMPTS[0], 8)])
        continuous_batching(wq, 1)
    n_tok, elapsed = continuous_batching(q, B)
    lat = [s.t_done - s.arrival for s in jobs]
    BATCH_SWEEP[B] = {"tok_s": n_tok / elapsed,
                      "mean_latency_s": float(np.mean(lat)),
                      "p50_latency_s": float(np.percentile(lat, 50))}
    print(f"{B:>10d} {n_tok / elapsed:>16.1f} "
          f"mean={np.mean(lat):>6.1f}  p50={np.percentile(lat, 50):>5.1f}")

RESULTS_SPEED["fp16 + continuous batching"] = {
    "tok_s_b1": BATCH_SWEEP[1]["tok_s"], "tok_s_b16": BATCH_SWEEP[16]["tok_s"]}
speedup = BATCH_SWEEP[16]["tok_s"] / BATCH_SWEEP[1]["tok_s"]
print(f"\nbatch 16 ได้ throughput รวม {speedup:.1f} เท่าของ batch 1")
print("แต่สังเกตคอลัมน์ latency ต่อ request: มันแย่ลง -- batching คือการ")
print("ขาย latency ของผู้ใช้แต่ละคน ซื้อ throughput ของระบบ ไม่ใช่ของฟรี")


### Stage 4 — Poisson load test: หา "หัวเข่า" ของ p99

ยิง request ตามตารางเวลา Poisson แบบ open-loop สามระดับความหนาแน่น
(ราว 30% / 60% / 90% ของ capacity ที่วัดได้ใน stage 3b) แล้วดู p50/p90/p99
ของ latency ต่อ request

สิ่งที่ควรเห็น: p50 แทบไม่ขยับ แต่ **p99 ระเบิดก่อนเครื่องเต็ม** — ที่ 90%
ของ capacity คิวเริ่มสะสมเป็นช่วง ๆ และหางของการแจกแจงยาวขึ้นมาก
ใครขนาดระบบให้พอดี 100% ของ throughput ที่วัดได้ กำลังออกแบบระบบที่ p99
เป็นอนันต์

(หมายเหตุความซื่อสัตย์: ที่ n=24 request ต่อระดับ ค่า p99 ในนิยามของ
`numpy.percentile` คือค่า max โดยพฤตินัย — รายงานไว้ตรง ๆ และนี่คือเหตุผลที่
production ต้องวัดด้วย request หลักหมื่นก่อนตั้ง SLO จริง)


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 4 -- open-loop Poisson load test -> p50/p90/p99 ต่ออัตราการมาถึง
# -----------------------------------------------------------------------------
N_REQ = 24                     # request ต่อระดับโหลด
LOAD_TOKENS = 48               # token ต่อคำตอบ (สั้นลงเพื่อให้จบในงบเวลา)
MAX_BATCH_SERVE = 16

cap_req_s = BATCH_SWEEP[16]["tok_s"] / LOAD_TOKENS   # capacity โดยประมาณ (req/s)
print(f"capacity โดยประมาณจาก stage 3b: {cap_req_s:.2f} req/s "
      f"(ที่ {LOAD_TOKENS} token/คำตอบ)\n")

rng = np.random.default_rng(SEED)
LOAD_RESULTS = {}
for frac in (0.3, 0.6, 0.9):
    lam = frac * cap_req_s
    gaps = rng.exponential(1.0 / lam, size=N_REQ)
    arrivals = np.cumsum(gaps)               # ตารางยิง -- ยึดตามนี้เคร่งครัด (open-loop)

    q = deque(Seq(PROMPTS[i % len(PROMPTS)], LOAD_TOKENS, arrival=float(arrivals[i]))
              for i in range(N_REQ))
    jobs = list(q)
    n_tok, elapsed = continuous_batching(q, MAX_BATCH_SERVE)

    lat = np.array([s.t_done - s.arrival for s in jobs])
    LOAD_RESULTS[round(lam, 3)] = {
        "utilisation": frac,
        "p50_s": float(np.percentile(lat, 50)),
        "p90_s": float(np.percentile(lat, 90)),
        "p99_s": float(np.percentile(lat, 99)),
    }
    print(f"lambda={lam:5.2f} req/s (~{100 * frac:.0f}% ของ capacity) | "
          f"p50={np.percentile(lat, 50):5.1f}s  p90={np.percentile(lat, 90):5.1f}s  "
          f"p99={np.percentile(lat, 99):5.1f}s")

print("\nสังเกตรูปทรง: p50 ขยับนิดเดียว แต่ p99 ที่ ~90% พุ่งหลายเท่า --")
print("นั่นคือ 'หัวเข่า' ของทฤษฎีคิว: ระบบที่ไม่มี headroom คือระบบที่ออกแบบมา")
print("ให้พังตอนคนใช้เยอะที่สุด")


### Stage 5 — Quantisation: int8 กับ nf4 และราคาที่แท้จริง

> **บาปต้นของ genre นี้: รายงานความเร็วโดยไม่รายงานคุณภาพ**
> ตัวเลข "nf4 ประหยัด VRAM 2 เท่า!" ที่ไม่มีคะแนนคุณภาพแนบมา ไม่ใช่ผลการทดลอง
> มันคือโฆษณา — เซลล์ถัดไปจึงรัน **TH-KNOW เต็มชุดกับทุก configuration**
> (fp16 วัดแล้วใน Cell 1) ด้วยสัญญาการวัดผลเดิมทุกตัวอักษร คอลัมน์คุณภาพ
> เป็นคอลัมน์บังคับของตารางสรุป ไม่ใช่ของแถม

คาดการณ์ล่วงหน้าอย่างตรงไปตรงมา: บน T4 **int8 ของ bitsandbytes มักจะ*ช้ากว่า*
fp16** — LLM.int8() แยก outlier ไปคูณใน fp16 ทำให้จ่าย overhead สองทาง
มันคือเครื่องมือประหยัด VRAM ไม่ใช่เครื่องมือเร่งความเร็ว ถ้าผลวัดของคุณ
ออกมาแบบนั้น นั่นไม่ใช่บั๊กของคุณ นั่นคือความจริงที่บทความรีวิวไม่ค่อยพิมพ์
ส่วน KV cache ไม่เกี่ยวกับ bitsandbytes เลย: มันยังเป็น fp16 ที่ 112 KiB/token เท่าเดิม


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 5 -- int8 / nf4: VRAM + tok/s + TH-KNOW (คอลัมน์คุณภาพเป็นคอลัมน์บังคับ)
# -----------------------------------------------------------------------------
from transformers import BitsAndBytesConfig

QUANT_RESULTS = {}
QUANT_CONFIGS = {
    "int8 (bitsandbytes)": BitsAndBytesConfig(load_in_8bit=True),
    "nf4 (bitsandbytes)": BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,   # T4: คำนวณใน fp16 เสมอ
    ),
}

# แถว fp16 อ้างอิง: ความเร็วจาก stage 2-3, คุณภาพจาก Cell 1
fp16_vram = WEIGHT_BYTES / 1e9
QUANT_RESULTS["fp16"] = {
    "vram_weights_gb": fp16_vram,
    "tok_s_b1": RESULTS_SPEED["fp16 naive generate"]["tok_s_b1"],
    "th_know_acc": FP16_QUALITY["accuracy"],
    "th_know_ci": [FP16_QUALITY["ci_low"], FP16_QUALITY["ci_high"]],
    "th_ratio": FP16_QUALITY["th_ratio"],
}

for name, qcfg in QUANT_CONFIGS.items():
    print(f"=== {name} ===")
    t0 = time.time()
    qm = AutoModelForCausalLM.from_pretrained(
        SERVE_DIR,
        quantization_config=qcfg,
        torch_dtype=DTYPE,             # โมดูลที่ไม่ถูก quantise (norm ฯลฯ) เป็น fp16
        attn_implementation=ATTN_IMPL,
        device_map={"": 0},
    )
    qm.eval()
    vram_gb = qm.get_memory_footprint() / 1e9
    print(f"  โหลดเสร็จใน {time.time() - t0:.0f}s | น้ำหนักบน VRAM ~{vram_gb:.2f} GB "
          f"(fp16 = {fp16_vram:.2f} GB)")

    tok_s_q, ttft_q, itl_q = timed_decode(qm, PROMPTS[0])
    print(f"  ความเร็ว: {tok_s_q:.1f} tok/s (batch 1) | "
          f"fp16 naive = {QUANT_RESULTS['fp16']['tok_s_b1']:.1f} tok/s")

    t0 = time.time()
    rep = evaluate(qm, tokenizer, slices=KOBEVAL_SLICES, seed=42,
                   model_name=f"Qwen3-0.6B {name}",
                   out_path=None, compute_ppl=False, progress=False)
    s = rep["slices"]["TH-KNOW"]
    print(f"  TH-KNOW: acc={s['accuracy']:.3f} [CI {s['ci_low']:.3f}-{s['ci_high']:.3f}] "
          f"th_ratio={s['th_ratio']:.2f}  (วัด {time.time() - t0:.0f}s)")

    QUANT_RESULTS[name] = {
        "vram_weights_gb": vram_gb, "tok_s_b1": tok_s_q,
        "th_know_acc": s["accuracy"], "th_know_ci": [s["ci_low"], s["ci_high"]],
        "th_ratio": s["th_ratio"],
    }
    del qm
    gc.collect()
    torch.cuda.empty_cache()

print("ครบสามแถว: fp16 / int8 / nf4 -- ทุกแถวมีทั้งความเร็ว, VRAM และคุณภาพ")
print("ห้ามสรุปจากคอลัมน์ความเร็วโดยไม่มองคอลัมน์ TH-KNOW เด็ดขาด")


---

## 8. ผลลัพธ์ (Results) — เขียน `results.json` ก่อนแตะ vLLM

รวมทุกตัวเลขที่วัดมา (stage 1-5) ลงไฟล์เดียว พร้อมวาดสองรูปหลักของตอน:

- **fig_roofline.png** — เพดานทฤษฎีจาก datasheet (เส้นประ) เทียบแท่งที่วัดจริง
  ทีละชั้นของ optimisation บวกเส้น throughput รวมตาม batch size
- **fig_latency_knee.png** — p50/p90/p99 ต่ออัตราการมาถึงจาก stage 4:
  หัวเข่าของทฤษฎีคิวในข้อมูลจริง

เซลล์นี้**ตั้งใจมาก่อน stage 6 (vLLM)** — เพราะ stage 6 เป็นเซลล์ที่เปราะที่สุด
ในซีรีส์ และผลของ stage 1-5 ต้องปลอดภัยอยู่บนดิสก์ก่อนเสี่ยง


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8 -- รูป roofline + latency knee แล้วเขียน results.json (ก่อน vLLM เสมอ)
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt

from kobeval import use_thai_font, write_results

use_thai_font()

# --- รูปที่ 1: roofline -----------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

stage_names = list(RESULTS_SPEED)
stage_vals = [RESULTS_SPEED[s]["tok_s_b1"] for s in stage_names]
axes[0].barh(stage_names, stage_vals, color="#2563eb")
axes[0].axvline(CEILING, ls="--", color="#dc2626", lw=2)
axes[0].text(CEILING * 0.99, 0.1, f"เพดาน datasheet ~{CEILING:.0f} tok/s",
             rotation=90, va="bottom", ha="right", color="#dc2626", fontsize=9)
for i, v in enumerate(stage_vals):
    axes[0].text(v + 2, i, f"{v:.0f}", va="center", fontsize=9)
axes[0].set_xlabel("tok/s ที่ batch = 1 (decode)")
axes[0].set_title("ปิดช่องว่างสู่เพดานทีละชั้น -- แต่ไม่มีวันแตะ", fontsize=11)
axes[0].grid(axis="x", alpha=0.3)
axes[0].set_axisbelow(True)

bs = sorted(BATCH_SWEEP)
axes[1].plot(bs, [BATCH_SWEEP[b]["tok_s"] for b in bs], "o-", color="#2563eb",
             lw=2, label="aggregate tok/s")
axes[1].plot(bs, [1000 * BATCH_SWEEP[b]["mean_latency_s"] for b in bs], "s--",
             color="#f59e0b", lw=2, label="mean latency (ms)")
axes[1].set_xlabel("batch size (continuous batching)")
axes[1].set_title("batching: ขาย latency ซื้อ throughput", fontsize=11)
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_axisbelow(True)

fig.tight_layout()
fig.savefig("fig_roofline.png", dpi=150, bbox_inches="tight")
plt.show()

# --- รูปที่ 2: latency knee ---------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(7.5, 4.4))
lams = sorted(LOAD_RESULTS)
for key, color, marker in [("p50_s", "#2563eb", "o"), ("p90_s", "#f59e0b", "s"),
                           ("p99_s", "#dc2626", "^")]:
    ax2.plot(lams, [LOAD_RESULTS[l][key] for l in lams], marker=marker, ls="-",
             color=color, lw=2, label=key.replace("_s", ""))
ax2.set_xlabel("อัตราการมาถึง lambda (req/s)")
ax2.set_ylabel("latency ต่อ request (วินาที)")
ax2.set_title("p99 ระเบิดก่อนเซิร์ฟเวอร์เต็ม -- หัวเข่าของทฤษฎีคิวในข้อมูลจริง",
              fontsize=11)
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_axisbelow(True)
fig2.tight_layout()
fig2.savefig("fig_latency_knee.png", dpi=150, bbox_inches="tight")
plt.show()

# --- เขียน results.json (ก่อนแตะ vLLM เสมอ) -----------------------------------
payload = {
    "post": 10,
    "slug": "llm-10-deployment",
    "notebook": "10_deployment.ipynb",
    "model": MODEL_ID,
    "gpu": GPU_NAME,
    "supports_bf16": SUPPORTS_BF16,
    "eval_contract": EVAL_CONTRACT,
    "theory": {
        "kv_bytes_per_token": KV_PER_TOKEN,
        "kv_full_context_gb": kv_full_ctx / 1e9,
        "weight_bytes": WEIGHT_BYTES,
        "t4_bandwidth_gb_s": T4_BW / 1e9,
        "decode_ceiling_tok_s": CEILING,
    },
    "serve_model": {"dir": SERVE_DIR, "adapter": adapter_path,
                    "adapter_bytes": adapter_bytes, "merged_bytes": serve_bytes},
    "measured": {
        "stages_b1": RESULTS_SPEED,
        "ratio_naive_vs_ceiling": RESULTS_SPEED["fp16 naive generate"]["tok_s_b1"] / CEILING,
        "batch_sweep": {str(k): v for k, v in BATCH_SWEEP.items()},
        "load_test": {str(k): v for k, v in LOAD_RESULTS.items()},
        "quantisation": QUANT_RESULTS,
    },
    "figures": ["fig_roofline.png", "fig_latency_knee.png"],
}
path = write_results(payload, "results.json")
print(f"เขียนไฟล์แล้ว: {path}  (ผล stage 1-5 ปลอดภัยแล้ว -- เสี่ยงกับ vLLM ได้)")


### Stage 6 — vLLM (ทางเลือก และเปราะที่สุดในทั้ง 10 โน้ตบุ๊ก)

> **เซลล์ถัดไปคือเซลล์ที่มีโอกาสพังสูงที่สุดในซีรีส์ — ด้วยเหตุผลที่อธิบายได้**
> vLLM บน T4 ฟรีของ Colab มีเงื่อนไขซ้อนกันหลายชั้น:
>
> 1. T4 คือ SM 7.5 (Turing) — **ต้องระบุ `dtype="half"`** เพราะเจอ bf16
>    ใน config ของ Qwen3 แล้ว vLLM จะปฏิเสธทันที (ถึงตอนนี้คุณควรทายได้เอง)
> 2. หลายเวอร์ชันต้อง **`enforce_eager=True`** เพราะเส้นทาง CUDA graph
>    มีปัญหาบนการ์ดเก่า และ engine รุ่น V1 ไม่รองรับ pre-Ampere —
>    เราจึงตั้ง `VLLM_USE_V1=0` ด้วย
> 3. บางเวอร์ชันล่าสุด**ตัดหรือพังการ build สำหรับ sm_75 ไปเลย** — เซลล์นี้จึง
>    **pin `vllm==0.8.5`** (เวอร์ชันแรก ๆ ที่รองรับสถาปัตยกรรม Qwen3) ห้ามอัปเป็น latest
> 4. การติดตั้ง vLLM **ดึง torch เวอร์ชันของมันเองมาทับ** torch ของ Colab —
>    สาเหตุอันดับหนึ่งที่ notebook พังตามคอมเมนต์ใน Cell 0 — จึงต้องทำเป็น
>    stage สุดท้ายหลัง `results.json` ถูกเขียนแล้วเท่านั้น และถ้าพัง
>    อาจต้อง Restart runtime แล้วรันใหม่โดยข้าม stage นี้

ทั้ง stage อยู่ใน try/except: ถ้า vLLM ติดตั้งไม่ผ่านหรือ crash ตอน init
ทุกอย่างใน stage 1-5 **ยังสมบูรณ์** — ข้อสรุปหลักของตอนไม่ได้พึ่ง vLLM เลย
มันเป็นแค่หลักฐานว่า paged KV + kernel ที่ fuse มาดี ทำอะไรได้อีก
เมื่อเทียบกับ scheduler 60 บรรทัดของเรา


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 6 (ทางเลือก) -- vLLM แบบ pin เวอร์ชัน + graceful skip รักษาผล stage 1-5
# -----------------------------------------------------------------------------
import subprocess
import sys

RUN_VLLM = True          # ตั้ง False เพื่อข้ามทั้ง stage โดยไม่เสี่ยงอะไรเลย
VLLM_PIN = "vllm==0.8.5"  # ห้ามอัปเป็น latest -- ดูเหตุผลข้อ 3 ใน markdown
VLLM_OK = False
VLLM_RESULTS = None

if RUN_VLLM:
    try:
        os.environ["VLLM_USE_V1"] = "0"      # engine V1 ไม่รองรับ pre-Ampere (sm_75)
        print(f"ติดตั้ง {VLLM_PIN} (อาจใช้เวลาหลายนาที และจะแตะ torch ของ Colab)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", VLLM_PIN],
                       check=True)

        # คืน VRAM ทั้งหมดให้ vLLM ก่อน init (merged กับ model คือ object เดียวกัน
        # ต้องปลดทุกชื่อที่ชี้อยู่ ไม่งั้นหน่วยความจำไม่ถูกคืนจริง)
        del model, fast, merged
        gc.collect()
        torch.cuda.empty_cache()

        from vllm import LLM, SamplingParams

        llm = LLM(
            model=SERVE_DIR,
            dtype="half",                 # T4 ไม่มี bf16 -- บังคับระบุ (มุกสุดท้ายของซีรีส์)
            enforce_eager=True,           # ข้าม CUDA graph ที่งอแงบน sm_75
            gpu_memory_utilization=0.85,
            max_model_len=2048,
        )
        sp = SamplingParams(temperature=0.0, max_tokens=64)

        llm.generate([PROMPTS[0]], sp)   # อุ่นเครื่องก่อนจับเวลา -- กติกาเดิม

        t0 = time.time()
        out1 = llm.generate([PROMPTS[0]], sp)
        t1 = time.time() - t0
        n1 = len(out1[0].outputs[0].token_ids)

        batch16 = [PROMPTS[i % len(PROMPTS)] for i in range(16)]
        t0 = time.time()
        out16 = llm.generate(batch16, sp)
        t16 = time.time() - t0
        n16 = sum(len(o.outputs[0].token_ids) for o in out16)

        VLLM_RESULTS = {"version": VLLM_PIN, "tok_s_b1": n1 / t1, "tok_s_b16": n16 / t16}
        VLLM_OK = True
        print(f"\nvLLM {VLLM_PIN}: batch1 = {n1 / t1:.1f} tok/s | "
              f"batch16 รวม = {n16 / t16:.1f} tok/s")
        print(f"เทียบ scheduler มือเขียนของเรา (batch16): "
              f"{BATCH_SWEEP[16]['tok_s']:.1f} tok/s")
        print("ถ้า vLLM ไม่ชนะชัดเจน แปลว่า enforce_eager กำลังกินกำไรของมัน --")
        print("บนการ์ด Ampere ขึ้นไปที่เปิด CUDA graph ได้ ช่องว่างจะกว้างกว่านี้มาก")

        # เติมผล vLLM ลง results.json (ของเดิมยังอยู่ครบ)
        payload["measured"]["vllm"] = VLLM_RESULTS
        write_results(payload, "results.json")
        print("อัปเดต results.json ด้วยผล vLLM แล้ว")
    except Exception as exc:
        print(f"\nvLLM ใช้ไม่ได้บนรันไทม์นี้ -- ข้ามได้เลย: {exc}")
        print("ผลทั้งหมดของ stage 1-5 อยู่ใน results.json เรียบร้อยแล้ว ไม่เสียอะไร")
        print("(ถ้า torch โดนทับจนรันเซลล์อื่นไม่ได้: Runtime > Restart แล้วรันใหม่")
        print(" โดยตั้ง RUN_VLLM = False)")
else:
    print("ข้าม stage 6 ตามที่ตั้งไว้ (RUN_VLLM = False)")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9 -- ตารางสรุปของทั้งตอน: ทุกความเร็วต้องโชว์ราคาที่จ่ายในแถวเดียวกัน
# -----------------------------------------------------------------------------
import pandas as pd

p99_at_09 = LOAD_RESULTS[max(LOAD_RESULTS)]["p99_s"]     # p99 ที่โหลด ~90%


def fmt_quality(name):
    q = QUANT_RESULTS.get(name)
    if q is None:
        return "-"
    lo, hi = q["th_know_ci"]
    return f"{q['th_know_acc']:.3f} [{lo:.3f}-{hi:.3f}]"


table_rows = [
    {"การตั้งค่า": "fp16 + naive generate",
     "tok/s @B=1": round(RESULTS_SPEED["fp16 naive generate"]["tok_s_b1"], 1),
     "tok/s @B=16": "-", "p99 (s)": "-",
     "VRAM (GB)": round(QUANT_RESULTS["fp16"]["vram_weights_gb"], 2),
     "TH-KNOW": fmt_quality("fp16")},
    {"การตั้งค่า": [k for k in RESULTS_SPEED if "static" in k][0],
     "tok/s @B=1": round([v for k, v in RESULTS_SPEED.items() if "static" in k][0]["tok_s_b1"], 1),
     "tok/s @B=16": "-", "p99 (s)": "-",
     "VRAM (GB)": round(QUANT_RESULTS["fp16"]["vram_weights_gb"], 2),
     "TH-KNOW": "= fp16 (น้ำหนักเดิมเป๊ะ)"},
    {"การตั้งค่า": "fp16 + continuous batching",
     "tok/s @B=1": round(BATCH_SWEEP[1]["tok_s"], 1),
     "tok/s @B=16": round(BATCH_SWEEP[16]["tok_s"], 1),
     "p99 (s)": round(p99_at_09, 1),
     "VRAM (GB)": round(QUANT_RESULTS["fp16"]["vram_weights_gb"], 2),
     "TH-KNOW": "= fp16 (น้ำหนักเดิมเป๊ะ)"},
    {"การตั้งค่า": "int8 (bitsandbytes)",
     "tok/s @B=1": round(QUANT_RESULTS["int8 (bitsandbytes)"]["tok_s_b1"], 1),
     "tok/s @B=16": "-", "p99 (s)": "-",
     "VRAM (GB)": round(QUANT_RESULTS["int8 (bitsandbytes)"]["vram_weights_gb"], 2),
     "TH-KNOW": fmt_quality("int8 (bitsandbytes)")},
    {"การตั้งค่า": "nf4 (bitsandbytes)",
     "tok/s @B=1": round(QUANT_RESULTS["nf4 (bitsandbytes)"]["tok_s_b1"], 1),
     "tok/s @B=16": "-", "p99 (s)": "-",
     "VRAM (GB)": round(QUANT_RESULTS["nf4 (bitsandbytes)"]["vram_weights_gb"], 2),
     "TH-KNOW": fmt_quality("nf4 (bitsandbytes)")},
]
if VLLM_OK and VLLM_RESULTS:
    table_rows.append(
        {"การตั้งค่า": f"vLLM {VLLM_RESULTS['version']} dtype=half",
         "tok/s @B=1": round(VLLM_RESULTS["tok_s_b1"], 1),
         "tok/s @B=16": round(VLLM_RESULTS["tok_s_b16"], 1),
         "p99 (s)": "-", "VRAM (GB)": "ตาม gpu_memory_utilization",
         "TH-KNOW": "= fp16 (น้ำหนักเดิมเป๊ะ)"})

df_final = pd.DataFrame(table_rows)
print(df_final.to_string(index=False))

payload["comparison_table"] = table_rows
write_results(payload, "results.json")
print("\nตารางสรุปถูกเขียนลง results.json แล้ว")
print("\nวิธีอ่านตาราง:")
print("- compile ช่วย batch 1 มากที่สุด (ฆ่า overhead ต่อ step ของ single-stream)")
print("- batching แทบไม่ช่วย tok/s ต่อ request แต่คูณ aggregate -- และแลก p99")
print("- int8 ลด VRAM แต่บน T4 มักช้าลง | nf4 ลด VRAM มากกว่าและมักเร็วกว่า int8")
print("- คุณภาพต้องดูคอลัมน์ TH-KNOW เอง ห้ามสรุปจากคอลัมน์ความเร็ว")


---

## 10. สรุป (Summary)

- **decode ที่ batch 1 คือการรอน้ำหนักเดินทาง ไม่ใช่รอการคำนวณ** — ชิปใช้งานจริง
  ~0.5% ของ FLOPS ขณะ "ทำงานเต็มที่"
- **เพดานความเร็วคำนวณได้จาก datasheet ก่อนรันโค้ด**: 320 GB/s ÷ ~1.2 GB ≈ 266 tok/s
  และตัวเลขที่วัดได้ทุกตัวต้องอ่านเทียบกับมัน
- **KV cache 112 KiB/token — assert จาก config จริง ไม่ใช่เลขที่จำมา**
  ที่ context เต็ม 40,960 token คือ ~4.7 GB ต่อ sequence: **context คือตัวกินงบ**
- **batching = ขาย latency ซื้อ throughput** และ continuous batching คือวิธีขาย
  ที่ขาดทุนน้อยที่สุด — scheduler 60 บรรทัดของเราพิสูจน์กลไกนี้ด้วยตัวเลขจริง
- **quantisation ต้องวัดคู่กับคุณภาพเสมอ** — int8 บน T4 ประหยัด VRAM แต่มักช้าลง
  และคอลัมน์ TH-KNOW คือส่วนที่บทความรีวิวชอบลืม
- **Little's law ผูกทุกอย่างเข้าด้วยกัน**: $L = \lambda W$ บอกจำนวน sequence
  ที่ต้องถือ ซึ่งย้อนกลับไปเป็นงบ KV ของสมการแรก
- **p99 ระเบิดก่อนเซิร์ฟเวอร์เต็ม** — ระบบที่ไม่มี headroom คือระบบที่ออกแบบมา
  ให้พังตอนคนใช้เยอะที่สุด

### ปิดซีรีส์

สิบตอนที่ผ่านมาคือทางเดินเส้นเดียว: ใส่ความรู้ → สอนรูปแบบ → จัด preference →
กลั่นให้เล็ก → กันพัง → วัดอย่างซื่อสัตย์ → เอาขึ้นเสิร์ฟแล้ววัดกับเพดานที่ฟิสิกส์กำหนด
สิ่งที่อยากให้ติดตัวไปไม่ใช่เทคนิคใดเทคนิคหนึ่ง แต่คือ**นิสัยสามข้อ**ที่ทุกตอนย้ำ:
ตั้งสมการก่อนเขียนโค้ด, วัดทุกอย่างพร้อม confidence interval,
และตีพิมพ์ข้อจำกัดของตัวเองไว้ท้ายบทเสมอ

โน้ตบุ๊กทั้งสิบตัวรันจบได้บน Colab ฟรี — อย่าเพิ่งเชื่อผม **ไปรันเอง**
แล้วดูว่าตัวเลขของคุณต่างจากของผมตรงไหน


---

## ข้อจำกัดของการทดลองนี้

เขียนไว้ตรง ๆ เพื่อไม่ให้ตัวเลขข้างบนถูกอ่านเกินกว่าที่มันบอกได้จริง

1. **T4 เป็นการ์ดปี 2018** — แบนด์วิดท์ 320 GB/s เทียบ ~3.35 TB/s ของ H100
   ตัวเลขสัมบูรณ์ทุกตัวในตอนนี้จึง**ไม่โอน**ไปเครื่องอื่น สิ่งที่โอนได้คือ
   *สมการและวิธีคิด* — เปลี่ยนค่าคงที่จาก datasheet ใบใหม่แล้วคำนวณซ้ำ

2. **การจับเวลาบน Colab มีสัญญาณรบกวน** — เครื่องเช่าร่วม (shared tenancy)
   ทำให้ตัวเลขแกว่งได้ ±10-20% ระหว่างรัน เราวัดครั้งเดียวต่อ configuration
   เพราะงบเวลา งานจริงควรวัดซ้ำหลายรอบแล้วรายงาน median พร้อมช่วง

3. **Load test ใช้ request น้อยมาก (24 ต่อระดับ)** — p99 ที่ n=24 คือค่า max
   โดยพฤตินัย รูปทรงหัวเข่าเชื่อถือได้ แต่ตัวเลข p99 สัมบูรณ์ไม่ควรอ้างอิง
   production ต้องวัดด้วย request หลักหมื่นก่อนตั้ง SLO

4. **Scheduler ของเรา prefill ซ้ำเมื่อสมาชิก batch เปลี่ยน** — จ่าย compute ซ้ำ
   ที่ vLLM ไม่ต้องจ่าย (paged KV) ตัวเลข continuous batching ของเราจึงเป็น
   **ขอบล่าง** ของสิ่งที่เทคนิคนี้ทำได้

5. **คุณภาพวัดด้วย TH-KNOW slice เดียว (30 ข้อ)** — CI กว้างราว ±17 จุด
   ความเสียหายจาก quantisation ที่เล็กกว่านั้นจะมองไม่เห็นในตารางนี้
   (แต่การ *มีคอลัมน์คุณภาพ* ก็ยังซื่อสัตย์กว่าการไม่วัดเลย)

6. **vLLM stage ขึ้นกับ ecosystem ภายนอก** — เวอร์ชันที่ pin ไว้ (0.8.5)
   อาจติดตั้งไม่ได้ในอนาคตเมื่อ Colab เปลี่ยน CUDA/Python และ SM 7.5
   กำลังถูกทยอยเลิกซัพพอร์ต — นี่คือเหตุผลที่มันเป็น stage ทางเลือก

7. **การเสิร์ฟจริงระดับ production ต้องการอีกหลายชั้นที่ตอนนี้ไม่ได้แตะ** —
   autoscaling, health check, observability, rate limiting, การยืนยันตัวตน,
   การจัดการเวอร์ชันโมเดลและ rollback — การไม่พูดถึงไม่ได้แปลว่าไม่สำคัญ
   แปลว่าโน้ตบุ๊กเดียวพูดได้ไม่หมด

---

*ซีรีส์นี้เผยแพร่ภายใต้สัญญาอนุญาต Apache-2.0 — [kobkrit.com](https://kobkrit.com)*
